# 01 · Prognosis is not treatment effect

**Left off (2026-10-11).** Open questions from Shaun:
- What exactly does "prognosis is not treatment effect" mean?
- Why do we need a toy example? Why not use the existing data instead of made-up data?

**Synthetic data.** Parts 1–2: made-up patients. Part 3: Kaggle CIBMTR data (also synthetic).

## Key

| Term | Meaning |
|---|---|
| HCT | Hematopoietic cell transplantation. Stem-cell (e.g. bone marrow) transplant. |
| CIBMTR | Center for International Blood and Marrow Transplant Research. Competition host. |
| EFS | Event-free survival. Event = relapse, graft failure or death. |
| Conditioning | Chemo and/or radiation before transplant. |
| MAC | Myeloablative conditioning. High dose. Better disease control, more toxic. |
| RIC | Reduced-intensity conditioning. Low dose. Less toxic. |
| DCA | Decision curve analysis. See Part 2. |
| p_t | Threshold probability. Risk above which you act. |
| GAN | Generative adversarial network. SurvivalGAN made the Kaggle data. |
| Cox | Cox proportional hazards model. Survival regression. A name, not an acronym. |
| Prognosis | Chance of a bad outcome for a patient. |
| Treatment effect | Change in that chance from treatment A to treatment B. |
| Risk model | Model that predicts prognosis. Also "prognostic model". |
| Confounder | Patient fact that changes both treatment choice and outcome. |
| Bias | Error in one direction. More data does not fix it. |
| Answer key | True risk with MAC and with RIC, per patient. Only made-up data has it. |

## Point

- Good risk model ≠ good treatment advice.
- Analogy: hospital patients die more than people at home. Cause: sick people go to hospital, not hospital harm.
- Doctors choose treatment for a reason (e.g. RIC for old, frail patients). So the RIC group does worse, even if RIC helps them.
- Our models take treatment as input (`conditioning_intensity`). Switch MAC → RIC, read new risk: looks like treatment advice. **Not reliable.**
- Model can correct only for recorded confounders. Doctors see unrecorded facts.

## Plan

| Part | Data | Shows |
|---|---|---|
| 1 | Made-up, answer key known | Risk model gives wrong treatment advice. |
| 2 | Same made-up | Same model still useful for act-or-not decision. |
| 3 | Kaggle CIBMTR | Same warning signs in real competition data. |

In [ ]:
from hct_equity import teaching
from hct_equity.config import load_config, load_labels
from hct_equity.data import load_train
from hct_equity.readable import readable
import dcurves

project = load_config()
settings = project.teaching_simulation
labels = load_labels()
print(project.synthetic_note)

## Part 1 · Made-up patients

Why: real patients get one treatment only. No answer key, so no check of treatment advice. Made-up patients: we write the rules, so we know both risks. Rules: `config/project.json` → `teaching_simulation`.

| Column | Meaning | Visible in real data? |
|---|---|---|
| `frail` | 1 = frail. Higher risk. | Yes |
| `severe` | 1 = aggressive disease. Higher risk. Doctor sees it. | **No** |
| `ric` | Doctor's choice: 1 = RIC, 0 = MAC. | Yes |
| `risk_mac`, `risk_ric` | True risk with each treatment. | No (answer key) |
| `event` | 1 = event, with the treatment given. | Yes |

Rules:
- Doctor: frail → usually RIC. Severe → usually MAC.
- Truth: frail → RIC better (MAC too toxic). Fit → MAC better (disease control).

Table: group means. `ric` = share given RIC.

In [ ]:
cohort = teaching.simulate_cohort(settings)
readable(cohort.groupby(["frail", "severe"])[["ric", "risk_mac", "risk_ric"]].mean(), labels)

Each row: one patient group. Columns: share given RIC, and true event risk with each treatment.

Matches the rules: frail → mostly RIC; severe → rarely RIC.

### Method 1: naive comparison

Event rate, RIC group vs MAC group. No correction.

In [ ]:
readable(cohort.groupby("ric")["event"].mean().rename("event_rate"), labels)

RIC looks **worse**. Cause: RIC group is mostly frail. Shows *who got RIC*, not *what RIC does*.

### Method 2: risk model

Logistic regression (formula → risk from 0 to 1):

> risk score = a + b × frail + c × ric + d × (frail × ric)

- Fit = computer picks a, b, c, d (coefficients) to match the data.
- `frail × ric`: lets RIC effect differ for frail vs fit.
- Model sees `frail`, `ric`. Not `severe`.

Treatment advice, 3 steps:
1. Set all patients to MAC → predicted risk.
2. Set all patients to RIC → predicted risk.
3. RIC − MAC. Negative = model says RIC helps.

In [ ]:
model = teaching.fit_prognostic(cohort, teaching.OBSERVED_FEATURES)
readable(teaching.effect_by_frailty(cohort, model, teaching.OBSERVED_FEATURES), labels)

Each value: event risk with RIC − event risk with MAC, in percentage points. Minus = RIC helps. Plus = RIC harms.

| Column | Source | Real data? |
|---|---|---|
| True effect of RIC | Answer key | No |
| Model's estimate | Method 2 | Yes |
| Naive comparison | Method 1, per frailty group | Yes |

- **Fit: wrong sign.** Truth: RIC harms. Model: RIC helps.
- Frail: right sign, too large.
- Model's estimate = naive comparison: 4 coefficients, 4 groups (fit/frail × MAC/RIC). Model just repeats group event rates.

Cause: `severe`. Table: number of fit patients, by treatment and severity.

In [ ]:
readable(cohort[cohort["frail"] == 0].groupby(["ric", "severe"]).size().unstack(), labels, cell_format="count")

- Nearly all severe fit patients got MAC (row MAC, column Severe).
- Severe → high risk with any treatment. So MAC group has more events.
- Model cannot see `severe` → blames MAC.
- This is **bias**: not chance (large sample), caused by the hidden `severe`.

### Choose treatment with the model?

| Policy | Rule |
|---|---|
| Current practice | Doctor's choice. |
| Everyone MAC / RIC | All get one treatment. |
| Choose with prognostic model | Lower model risk. |
| Choose with true risks | Lower true risk. Best possible. |

Average event risk: true mean risk under the policy (lower = better; needs answer key). Patients given RIC: share of all patients.

In [ ]:
readable(teaching.compare_policies(cohort, model, teaching.OBSERVED_FEATURES), labels)

- Model policy: **RIC for all**. Worse than current practice.
- Doctors see `severe`, so they choose better.

### Check: add `severe` to the model

In [ ]:
full_model = teaching.fit_prognostic(cohort, teaching.ALL_FEATURES)
display(readable(teaching.effect_by_frailty(cohort, full_model, teaching.ALL_FEATURES), labels))
readable(teaching.compare_policies(cohort, full_model, teaching.ALL_FEATURES), labels)

- Model's estimate ≈ true effect. Model policy = best policy.
- So hidden `severe` caused the error.

**Lesson:** correct treatment advice needs **every** confounder in the data ("no unmeasured confounding"). Data cannot test this. It is an assumption.

## Part 2 · What DCA can answer

Act-or-not decision. Example: extra check-ups after transplant. Cheap, safe.
- Model only needs to rank patients by risk. Not *why*.
- So the biased Part 1 model can still help.

DCA compares:
- **Treat all**: act for everyone.
- **Treat none**: act for no one.
- **Model**: act if predicted risk > p_t.

> p_t / (1 − p_t) = harm of unnecessary action ÷ benefit of necessary action

- p_t = 0.1 → 1/9: accept up to 9 unnecessary check-ups per necessary one.
- Net benefit = necessary actions − unnecessary actions × p_t / (1 − p_t), per patient. Higher = better.

In [ ]:
thresholds = project.teaching_dca_thresholds.values()
dca_result = teaching.surveillance_dca(cohort, model, teaching.OBSERVED_FEATURES, thresholds)
dcurves.plot_graphs(plot_df=dca_result, graph_type="net_benefit", y_limits=(-0.05, 0.7))

Plot: x = p_t, y = net benefit.
- Treat none = 0.
- Treat all high at low p_t: ~59% of made-up patients have an event.
- Low p_t: all predicted risks > p_t → model = treat all.
- Higher p_t: model above both. Model useful there.

**Same model: useful for check-ups, harmful for treatment choice.**

CIBMTR: ~52% event by 12 months. Treat all strong at low p_t; our models must beat it.

| Decision | Model must get right | Risk model OK? | Check with |
|---|---|---|---|
| Act or not (check-up, referral, trial screen) | Who is high risk | Yes | DCA |
| Treatment A vs B (MAC/RIC, donor) | Risk **change** per patient | Only if no unmeasured confounding | Randomised trial or causal methods |

- Randomised trial: random treatment → no confounding.
- Causal methods (target-trial emulation, propensity methods): copy a trial with recorded data. Same assumption.
- DCA: one action vs none. "A instead of B" is not one action. DCA of a treatment-benefit model: only after a causal estimate.

## Part 3 · CIBMTR data

No answer key. Outcome only for the treatment given.

Table, per arm: mean age at HCT, mean comorbidity score (other illnesses; higher = sicker), mean Karnofsky score (daily function, 0–100; higher = fitter), patients, event rate (any time).

In [ ]:
example = project.treatment_example
train = load_train(project)
readable(teaching.arm_case_mix(train, example.column, example.arms, example.case_mix, project.columns.event), labels)

- RIC: more events. Also older, sicker, less fit. Same pattern as Part 1 → naive comparison confounded.
- Model corrects for recorded age, comorbidity, Karnofsky. Not for unrecorded disease status, fitness detail, patient wishes.
- Synthetic (SurvivalGAN): any effect = what the generator learned, not a medical fact.

## For the CIBMTR summary

1. Evaluate models as **risk** models: discrimination (risk ranking), calibration by race group (predicted vs actual rates), DCA (act-or-not).
2. No claim on best conditioning, donor or graft.
3. Limits section must say this. Treatment comparison needs a causal method.